# ChatGPT Generated

In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import IO, List, Optional, Union
import io
import xml.etree.ElementTree as ET
from pathlib import Path

# ----------------------------
# Data model
# ----------------------------

@dataclass(frozen=True, slots=True)
class Title:
    text: str
    type: Optional[str] = None
    short: Optional[str] = None


@dataclass(frozen=True, slots=True)
class TextNode:
    text: str


@dataclass(frozen=True, slots=True)
class TransChangeNode:
    change_type: Optional[str]
    text: str


Node = Union[TextNode, TransChangeNode]


@dataclass(slots=True)
class Verse:
    osis_id: str
    n: Optional[str] = None
    sid: Optional[str] = None
    titles: List[Title] = field(default_factory=list)
    nodes: List[Node] = field(default_factory=list)

    def add_text(self, s: str) -> None:
        if not s:
            return
        if self.nodes and isinstance(self.nodes[-1], TextNode):
            self.nodes[-1] = TextNode(self.nodes[-1].text + s)
        else:
            self.nodes.append(TextNode(s))

    def add_trans_change(self, change_type: Optional[str], text: str) -> None:
        self.nodes.append(TransChangeNode(change_type=change_type, text=text))


@dataclass(slots=True)
class Paragraph:
    verses: List[Verse] = field(default_factory=list)


@dataclass(slots=True)
class Chapter:
    osis_ref: Optional[str] = None  # e.g. "Gen.1"
    n: Optional[str] = None         # e.g. "1"
    sid: Optional[str] = None       # opening sID (close marker uses eID==sid)
    titles: List[Title] = field(default_factory=list)
    paragraphs: List[Paragraph] = field(default_factory=list)


@dataclass(slots=True)
class Book:
    osis_id: str                    # <div type="book" osisID="Gen">
    titles: List[Title] = field(default_factory=list)
    chapters: List[Chapter] = field(default_factory=list)


# ----------------------------
# Parser
# ----------------------------

def _norm_ws(s: str) -> str:
    return " ".join(s.split())


def parse_osis_books(
    source: Union[str, IO[str], IO[bytes]],
    *,
    normalize_whitespace: bool = True,
) -> List[Book]:
    """
    Extract:
      books -> chapters -> paragraphs -> verses -> nodes

    Book:
      - opens on:  <div type="book" osisID="Gen" ...>
      - closes on: </div> for that book div

    Chapter (OSIS markers):
      - opens on:  <chapter ... sID="X" .../>
      - closes on: <chapter eID="X" />

    Titles:
      - <title> is accumulated into pending_titles and attached to the next
        Book or Chapter or Verse encountered (in that order of occurrence).
    """
    if isinstance(source, str) and "<" in source and "\n" in source:
        stream: Union[IO[str], IO[bytes]] = io.StringIO(source)
    else:
        stream = source

    books: List[Book] = []

    cur_book: Optional[Book] = None

    cur_chapter: Optional[Chapter] = None
    cur_chapter_open_id: Optional[str] = None  # sID

    cur_paragraph: Optional[Paragraph] = None

    cur_verse: Optional[Verse] = None
    cur_verse_sid: Optional[str] = None

    pending_titles: List[Title] = []

    # stack of div types so we can detect the closing </div> for a book div
    div_type_stack: List[Optional[str]] = []

    def _norm(s: str) -> str:
        return _norm_ws(s) if normalize_whitespace else s

    def attach_pending_titles(dst: List[Title]) -> None:
        nonlocal pending_titles
        if pending_titles:
            dst.extend(pending_titles)
            pending_titles = []

    def ensure_book() -> Book:
        nonlocal cur_book
        if cur_book is None:
            # If caller parses fragments without explicit <div type="book">, create a sentinel.
            cur_book = Book(osis_id="")
            books.append(cur_book)
        return cur_book

    def ensure_chapter() -> Chapter:
        nonlocal cur_chapter
        if cur_chapter is None:
            cur_chapter = Chapter()
            ensure_book().chapters.append(cur_chapter)
        return cur_chapter

    def ensure_paragraph() -> Paragraph:
        nonlocal cur_paragraph
        ch = ensure_chapter()
        if cur_paragraph is None:
            cur_paragraph = Paragraph()
            ch.paragraphs.append(cur_paragraph)
        return cur_paragraph

    def add_text_to_current_verse(text: Optional[str]) -> None:
        if not cur_verse or not text:
            return
        if normalize_whitespace:
            cleaned = _norm_ws(text)
            if cleaned:
                cur_verse.add_text(cleaned + " ")
        else:
            cur_verse.add_text(text)

    ctx = ET.iterparse(stream, events=("start", "end"))

    for event, elem in ctx:
        tag = elem.tag

        if event == "start":
            # print(f"Found <${tag}/>")
            if tag == "div":
                div_type = elem.attrib.get("type")
                div_type_stack.append(div_type)

                if div_type == "book":
                    osis_id = elem.attrib.get("osisID")
                    if not osis_id:
                        # If malformed, skip creating book (or set to empty).
                        osis_id = ""

                    # Reset nested state when a new book starts
                    cur_book = Book(osis_id=osis_id)
                    attach_pending_titles(cur_book.titles)
                    books.append(cur_book)

                    cur_chapter = None
                    cur_chapter_open_id = None
                    cur_paragraph = None
                    cur_verse = None
                    cur_verse_sid = None

            elif tag == "chapter":
                # open marker uses sID (close marker uses eID with same value)
                sid = elem.attrib.get("sID")
                if sid is not None:
                    cur_paragraph = None
                    cur_verse = None
                    cur_verse_sid = None

                    cur_chapter = Chapter(
                        osis_ref=elem.attrib.get("osisRef") or elem.attrib.get("osisID"),
                        n=elem.attrib.get("n"),
                        sid=sid,
                    )
                    cur_chapter_open_id = sid
                    attach_pending_titles(cur_chapter.titles)
                    ensure_book().chapters.append(cur_chapter)

            elif tag == "p":
                cur_paragraph = Paragraph()
                ensure_chapter().paragraphs.append(cur_paragraph)

            elif tag == "verse" and "osisID" in elem.attrib:
                p = ensure_paragraph()
                cur_verse = Verse(
                    osis_id=elem.attrib["osisID"],
                    n=elem.attrib.get("n"),
                    sid=elem.attrib.get("sID"),
                )
                attach_pending_titles(cur_verse.titles)
                p.verses.append(cur_verse)
                cur_verse_sid = elem.attrib.get("sID")

        else:  # event == "end"
            if tag == "title":
                title_text = _norm("".join(elem.itertext()))
                if title_text:
                    pending_titles.append(
                        Title(
                            text=title_text,
                            type=elem.attrib.get("type"),
                            short=elem.attrib.get("short"),
                        )
                    )

            elif tag == "chapter":
                # close marker uses eID == opening sID
                eid = elem.attrib.get("eID")
                if eid is not None and cur_chapter_open_id is not None and eid == cur_chapter_open_id:
                    cur_chapter = None
                    cur_chapter_open_id = None
                    cur_paragraph = None
                    cur_verse = None
                    cur_verse_sid = None

            elif tag == "verse":
                if "osisID" in elem.attrib:
                    add_text_to_current_verse(elem.tail)

                if "eID" in elem.attrib:
                    if cur_verse_sid is None or elem.attrib["eID"] == cur_verse_sid:
                        cur_verse = None
                        cur_verse_sid = None

            elif tag == "transChange":
                if cur_verse is not None:
                    change_type = elem.attrib.get("type")
                    inner_text = _norm("".join(elem.itertext()))
                    if inner_text:
                        cur_verse.add_trans_change(change_type, inner_text)
                    add_text_to_current_verse(elem.tail)

            elif tag == "p":
                cur_paragraph = None

            elif tag == "div":
                # closing a div: pop stack and if it was a book div, close the book context
                if div_type_stack:
                    closed_type = div_type_stack.pop()
                else:
                    closed_type = None

                if closed_type == "book":
                    cur_book = None
                    cur_chapter = None
                    cur_chapter_open_id = None
                    cur_paragraph = None
                    cur_verse = None
                    cur_verse_sid = None

            elem.clear()

    # Clean trailing space injected by normalize_whitespace=True
    if normalize_whitespace:
        for b in books:
            for ch in b.chapters:
                for p in ch.paragraphs:
                    for v in p.verses:
                        if v.nodes and isinstance(v.nodes[-1], TextNode):
                            v.nodes[-1] = TextNode(v.nodes[-1].text.rstrip())

    return books


In [2]:

xml_data = Path("eng-kjv.osis.xml").read_text(encoding="utf-8")
# xml_data = Path("snippet.xml").read_text(encoding="utf-8")


In [3]:
books = parse_osis_books(xml_data, normalize_whitespace=True)

In [4]:
len(books)

82

In [5]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any

@dataclass(frozen=True, slots=True)
class Section:
    id: int
    book: str
    start_chapter: int
    start_verse: int
    end_chapter: int
    end_verse: int
    title: str
    summary: str

def load_sections(path: str | Path) -> list[Section]:
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        data: Any = json.load(f)

    if not isinstance(data, list):
        raise TypeError(f"Expected top-level JSON array, got {type(data).__name__}")

    return [Section(**obj) for obj in data]  # requires keys match field names


In [6]:
sections = load_sections("kjv_pericopes.json")

In [7]:
osis_name_by_pericope_name = {
'Act': 'Acts',
'Amo': 'Amos',
'Ch1': '1Chr',
'Ch2': '2Chr',
'Co1': '1Cor',
'Co2': '2Cor',
'Col': 'Col',
'Dan': 'Dan',
'Deu': 'Deut',
'Ecc': 'Eccl',
'Eph': 'Eph',
'Est': 'Esth',
'Exo': 'Exod',
'Eze': 'Ezek',
'Ezr': 'Ezra',
'Gal': 'Gal',
'Gen': 'Gen',
'Hab': 'Hab',
'Hag': 'Hag',
'Heb': 'Heb',
'Hos': 'Hos',
'Isa': 'Isa',
'Jam': 'Jas',
'Jde': 'Jude',
'Jdg': 'Judg',
'Jer': 'Jer',
'Jo1': '1John',
'Jo2': '2John',
'Jo3': '3John',
'Job': 'Job',
'Joe': 'Joel',
'Joh': 'John',
'Jon': 'Jonah',
'Jos': 'Josh',
'Kg1': '1Kgs',
'Kg2': '2Kgs',
'Lam': 'Lam',
'Lev': 'Lev',
'Luk': 'Luke',
'Mal': 'Mal',
'Mar': 'Mark',
'Mat': 'Matt',
'Mic': 'Mic',
'Nah': 'Nah',
'Neh': 'Neh',
'Num': 'Num',
'Oba': 'Obad',
'Pe1': '1Pet',
'Pe2': '2Pet',
'Phi': 'Phil',
'Plm': 'Phlm',
'Pro': 'Prov',
'Psa': 'Ps',
'Rev': 'Rev',
'Rom': 'Rom',
'Rut': 'Ruth',
'Sa1': '1Sam',
'Sa2': '2Sam',
'Sol': 'Song',
'Th1': '1Thess',
'Th2': '2Thess',
'Ti1': '1Tim',
'Ti2': '2Tim',
'Tit': 'Titus',
'Zac': 'Zech',
'Zep': 'Zeph'
}

pericope_name_by_osis_name = {
'Acts': 'Act', 
'Amos': 'Amo', 
'1Chr': 'Ch1', 
'2Chr': 'Ch2', 
'1Cor': 'Co1', 
'2Cor': 'Co2', 
'Col': 'Col', 
'Dan': 'Dan', 
'Deut': 'Deu', 
'Eccl': 'Ecc', 
'Eph': 'Eph', 
'Esth': 'Est', 
'Exod': 'Exo', 
'Ezek': 'Eze', 
'Ezra': 'Ezr', 
'Gal': 'Gal', 
'Gen': 'Gen', 
'Hab': 'Hab', 
'Hag': 'Hag', 
'Heb': 'Heb', 
'Hos': 'Hos', 
'Isa': 'Isa', 
'Jas': 'Jam', 
'Jude': 'Jde', 
'Judg': 'Jdg', 
'Jer': 'Jer', 
'1John': 'Jo1', 
'2John': 'Jo2', 
'3John': 'Jo3', 
'Job': 'Job', 
'Joel': 'Joe', 
'John': 'Joh', 
'Jonah': 'Jon', 
'Josh': 'Jos', 
'1Kgs': 'Kg1', 
'2Kgs': 'Kg2', 
'Lam': 'Lam', 
'Lev': 'Lev', 
'Luke': 'Luk', 
'Mal': 'Mal', 
'Mark': 'Mar', 
'Matt': 'Mat', 
'Mic': 'Mic', 
'Nah': 'Nah', 
'Neh': 'Neh', 
'Num': 'Num', 
'Obad': 'Oba', 
'1Pet': 'Pe1', 
'2Pet': 'Pe2', 
'Phil': 'Phi', 
'Phlm': 'Plm', 
'Prov': 'Pro', 
'Ps': 'Psa', 
'Rev': 'Rev', 
'Rom': 'Rom', 
'Ruth': 'Rut', 
'1Sam': 'Sa1', 
'2Sam': 'Sa2', 
'Song': 'Sol', 
'1Thess': 'Th1', 
'2Thess': 'Th2', 
'1Tim': 'Ti1', 
'2Tim': 'Ti2', 
'Titus': 'Tit', 
'Zech': 'Zac', 
'Zeph': 'Zep'
}

In [8]:
import re

_REF_RE = re.compile(r"^(?P<book>[A-Za-z1-3]+)\.?(?P<chapter>\d+)\.(?P<verse>\d+)$")

def parse_ref(s: str) -> dict:
    m = _REF_RE.match(s)
    if not m:
        raise ValueError(f"Invalid reference: {s!r}")
    return {
        "book": m.group("book"),
        "chapter": int(m.group("chapter")),
        "verse": int(m.group("verse")),
    }

for book in books:
    for chapter in book.chapters:
        for paragraph in chapter.paragraphs:
            for verse in paragraph.verses:
                ref = parse_ref(verse.osis_id)
                pericope_title = next((section.title for section in sections if osis_name_by_pericope_name[section.book] == ref['book'] and section.start_chapter == ref['chapter'] and section.start_verse == ref['verse']), None)
                if(pericope_title):
                    print(f"{ref}: {pericope_title}")
                    verse.titles.append(Title(pericope_title, "AI Generated", None))

{'book': 'Gen', 'chapter': 1, 'verse': 1}: Creation of the world
{'book': 'Gen', 'chapter': 1, 'verse': 23}: Creation
{'book': 'Gen', 'chapter': 2, 'verse': 1}: Creation: Adam
{'book': 'Gen', 'chapter': 2, 'verse': 23}: Adam and Woman
{'book': 'Gen', 'chapter': 3, 'verse': 1}: Adam and Yea
{'book': 'Gen', 'chapter': 3, 'verse': 23}: Eden and Cherubims
{'book': 'Gen', 'chapter': 4, 'verse': 1}: Cain and Abel
{'book': 'Gen', 'chapter': 4, 'verse': 23}: Lamech and Cain
{'book': 'Gen', 'chapter': 5, 'verse': 1}: Genealogy of Adam
{'book': 'Gen', 'chapter': 5, 'verse': 8}: Genealogy of Enoch
{'book': 'Gen', 'chapter': 5, 'verse': 30}: Genealogy of Noah
{'book': 'Gen', 'chapter': 6, 'verse': 1}: Creation: Noah
{'book': 'Gen', 'chapter': 7, 'verse': 1}: The Ark: Noah
{'book': 'Gen', 'chapter': 7, 'verse': 10}: The Ark: Noah
{'book': 'Gen', 'chapter': 8, 'verse': 1}: The Ark: Noah
{'book': 'Gen', 'chapter': 8, 'verse': 13}: The Ark: Noah
{'book': 'Gen', 'chapter': 9, 'verse': 1}: Covenant: Noa

In [9]:
import json
import dataclasses

with open("output.json", "w") as f:
    f.write(json.dumps([ dataclasses.asdict(b) for b in books], indent=2))